<img src="../public/colorlogo.png" width="50%"/>

[Homepage](https://www.datafog.ai) | 
[Discord](https://discord.gg/bzDth394R4) | 
[Github](https://github.com/datafog/datafog-python) | 
[Contact](mailto:sid@datafog.ai) |
[Documentation](https://www.datafog.ai/datafog-docs/)

# DataFog Quick Start Guide\n\n> **📦 Version Requirement**: This guide is current for DataFog v4.8.0 and above\n> \n> ✅ **4.x Highlights**: GLiNER integration, smart cascading, agent guardrails, allowlists, and redaction correctness fixes\n\nWelcome to DataFog! This notebook demonstrates how to get started with DataFog's fast PII detection and anonymization capabilities.\n\n## What makes DataFog special?\n\n- **🚀 Ultra-Fast**: 190x faster than spaCy for structured PII, 32x faster with GLiNER\n- **🪶 Lightweight**: <2MB core package with optional ML extras\n- **🧠 Smart Engines**: Choose from regex, GLiNER, spaCy, or smart cascading\n- **📦 Production Ready**: Comprehensive testing and performance validation"

## Installation

Let's start by installing DataFog with the advanced features:

In [ ]:
# Install DataFog with advanced ML features
!pip install datafog[nlp-advanced] --quiet

print("✅ DataFog installed successfully!")

## 1. Simple API - Get Started in Seconds

The fastest way to detect PII in your text:

In [ ]:
from datafog import DataFog

# Create a DataFog instance
detector = DataFog()

# Sample text with various PII types
sample_text = """
Hi there! I'm Dr. Sarah Johnson, and you can reach me at sarah.johnson@hospital.com 
or call my office at (555) 123-4567. My SSN is 123-45-6789 for verification.
I work at General Hospital located at 123 Main St, New York, NY 10001.
My credit card ending in 4111-1111-1111-1111 expires on 12/25.
"""

# Detect PII - this uses the fast regex engine by default
results = detector.scan_text(sample_text)

print("🔍 PII Detection Results:")
print(f"Found {len(results)} pieces of PII:")
for entity_type, entities in results.items():
    if entities:  # Only show types that were found
        print(f"  {entity_type}: {entities}")

## 2. Engine Comparison - Choose Your Power Level

DataFog offers multiple engines for different needs. Let's compare them:

In [ ]:
from datafog.services import TextService
import time

# Test text with both structured and unstructured PII
test_text = "Dr. John Smith works at General Hospital. Contact him at john@hospital.com or (555) 123-4567."

# Engine configurations
engines = {
    "regex": "🚀 Fastest - Pattern-based detection",
    "gliner": "⚡ Fast - Modern ML with high accuracy", 
    "smart": "🧠 Balanced - Combines regex + GLiNER for best results"
}

print("⚡ Engine Performance Comparison\n")

for engine_name, description in engines.items():
    try:
        print(f"{description}")
        
        # Create service with specific engine
        service = TextService(engine=engine_name)
        
        # Time the detection
        start_time = time.time()
        result = service.annotate_text_sync(test_text)
        end_time = time.time()
        
        # Show results
        processing_time = (end_time - start_time) * 1000  # Convert to milliseconds
        print(f"  ⏱️  Processing time: {processing_time:.2f}ms")
        print(f"  🎯 Entities found: {list(result.keys()) if result else 'None'}")
        print()
        
    except ImportError as e:
        print(f"  ❌ {engine_name} engine not available (missing dependencies)")
        print(f"     Install with: pip install datafog[nlp-advanced]")
        print()
    except Exception as e:
        print(f"  ⚠️  Error with {engine_name}: {str(e)}")
        print()

## 3. Advanced Detection with GLiNER

GLiNER is DataFog's modern ML engine that provides excellent accuracy for named entity recognition:

In [ ]:
# Complex text with various entity types
complex_text = """
Medical Report - Patient: Emily Rodriguez, DOB: 03/15/1985
Dr. Michael Chen from Stanford Medical Center treated the patient.
Insurance ID: INS-789-456-123, Policy expires December 2024.
Emergency contact: Maria Rodriguez at (408) 555-9876.
Address: 1234 Oak Street, San Francisco, CA 94102
Lab results show glucose level of 120 mg/dL on 2024-01-15.
"""

try:
    # Use GLiNER for advanced entity detection
    gliner_service = TextService(engine="gliner")
    
    print("🧠 GLiNER Advanced Detection Results:")
    print("=" * 50)
    
    results = gliner_service.annotate_text_sync(complex_text)
    
    for entity_type, entities in results.items():
        if entities:  # Only show found entities
            print(f"\n{entity_type}:")
            for entity in entities:
                print(f"  • {entity}")
    
    print(f"\n✅ Total entity types detected: {len([k for k, v in results.items() if v])}")
    
except ImportError:
    print("❌ GLiNER not available. Install with: pip install datafog[nlp-advanced]")
except Exception as e:
    print(f"⚠️  GLiNER error: {e}")
    print("Falling back to regex engine...")
    
    # Fallback to regex
    regex_service = TextService(engine="regex")
    results = regex_service.annotate_text_sync(complex_text)
    print("\n🚀 Regex Detection Results:")
    for entity_type, entities in results.items():
        if entities:
            print(f"  {entity_type}: {entities}")

## 4. Smart Cascading - Best of All Worlds

The "smart" engine combines regex speed with GLiNER accuracy by using a cascading approach:

In [ ]:
# Text mixing structured PII (perfect for regex) and entities (better with GLiNER)
mixed_text = """
From: john.doe@techcorp.com
To: legal@company.com
Subject: Employee Data Update

Dear Legal Team,

Please update the employee record for Sarah Williams (ID: EMP-12345).
Her new phone number is (555) 987-6543 and SSN is 987-65-4321.
She works at our Seattle office and reports to Manager David Chen.
Her emergency contact is her spouse, Michael Williams, at (555) 111-2222.

Best regards,
HR Department
"""

try:
    # Smart engine: Uses regex first (fast), then GLiNER for missed entities
    smart_service = TextService(engine="smart")
    
    print("🧠 Smart Cascading Detection:")
    print("=" * 40)
    print("Strategy: Regex (speed) → GLiNER (accuracy)\n")
    
    start_time = time.time()
    results = smart_service.annotate_text_sync(mixed_text)
    end_time = time.time()
    
    # Organize results by category
    structured_pii = ['EMAIL', 'PHONE', 'SSN', 'CREDIT_CARD']
    entity_pii = ['PERSON', 'ORG', 'LOC', 'DATE_TIME']
    
    print("📧 Structured PII (Regex-optimized):")
    for entity_type in structured_pii:
        if entity_type in results and results[entity_type]:
            print(f"  {entity_type}: {results[entity_type]}")
    
    print("\n👤 Named Entities (GLiNER-optimized):")
    for entity_type in entity_pii:
        if entity_type in results and results[entity_type]:
            print(f"  {entity_type}: {results[entity_type]}")
    
    processing_time = (end_time - start_time) * 1000
    print(f"\n⏱️  Total processing time: {processing_time:.2f}ms")
    print(f"✅ Combined detection power with optimized speed!")
    
except Exception as e:
    print(f"⚠️  Smart engine error: {e}")
    print("This usually means GLiNER dependencies are missing.")
    print("Install with: pip install datafog[nlp-advanced]")

## 5. Anonymization - Protect Your Data

DataFog doesn't just detect PII - it can also anonymize it in various ways:

In [ ]:
# Sample sensitive data
sensitive_data = """
Patient: John Smith
Email: john.smith@email.com
Phone: (555) 123-4567
SSN: 123-45-6789
Credit Card: 4111-1111-1111-1111
Address: 123 Main St, Anytown, NY 12345
"""

print("🔒 DataFog Anonymization Methods\n")
print("Original text:")
print(sensitive_data)
print("=" * 60)

# Method 1: Redaction (replace with [REDACTED])
redactor = DataFog(operations=["scan", "redact"])
redacted_text = redactor.process_text(sensitive_data)
print("\n🚫 REDACTED:")
print(redacted_text)

# Method 2: Replacement (replace with fake but realistic data)
replacer = DataFog(operations=["scan", "replace"])
replaced_text = replacer.process_text(sensitive_data)
print("\n🔄 REPLACED:")
print(replaced_text)

# Method 3: Hashing (one-way transformation)
from datafog.models.anonymizer import HashType
hasher = DataFog(
    operations=["scan", "hash"],
    hash_type=HashType.SHA256
)
hashed_text = hasher.process_text(sensitive_data)
print("\n#️⃣ HASHED (SHA256):")
print(hashed_text)

## 6. Selective Processing - Target Specific PII Types

Sometimes you only want to process certain types of PII:

In [ ]:
# Sample text with mixed PII
business_data = """
Company Report:
CEO: Amanda Johnson (amanda@company.com)
CFO: Robert Davis (robert.davis@company.com) 
Phone: (555) 100-2000
Headquarters: 456 Business Ave, Corporate City, CA 90210
Tax ID: 12-3456789
Employee SSN for payroll: 987-65-4321
"""

print("🎯 Selective PII Processing\n")
print("Original text:")
print(business_data)
print("=" * 50)

# Only process emails and SSNs, leave names and addresses
selective_redactor = DataFog(
    operations=["scan", "redact"],
    entities=["EMAIL", "SSN"]  # Only target these types
)

selective_result = selective_redactor.process_text(business_data)
print("\n🎯 Selective Redaction (EMAIL + SSN only):")
print(selective_result)
print("\n💡 Notice: Names and addresses are preserved!")

## 7. Batch Processing - Handle Multiple Documents

Process multiple documents efficiently:

In [ ]:
# Sample document collection
documents = [
    "Patient file 1: John Doe, DOB: 01/15/1980, Phone: (555) 111-1111",
    "Customer record: jane@email.com, Account: 4532-1234-5678-9012", 
    "Employee data: Robert Smith, SSN: 123-45-6789, Manager: Sarah Lee",
    "Contact info: michael@company.com, Office: (555) 999-8888",
    "Invoice #1234: Bill to John at 123 Oak St, Los Angeles, CA 90001"
]

print("📚 Batch Processing Demo\n")
print(f"Processing {len(documents)} documents...\n")

# Process all documents at once
batch_detector = DataFog()
start_time = time.time()
batch_results = batch_detector.batch_process(documents)
end_time = time.time()

# Summary results
total_entities = 0
entity_counts = {}

for i, result in enumerate(batch_results):
    print(f"📄 Document {i+1}:")
    doc_entities = 0
    for entity_type, entities in result.items():
        if entities:
            count = len(entities)
            doc_entities += count
            entity_counts[entity_type] = entity_counts.get(entity_type, 0) + count
            print(f"  {entity_type}: {entities}")
    
    if doc_entities == 0:
        print("  No PII detected")
    total_entities += doc_entities
    print()

# Performance summary
processing_time = (end_time - start_time) * 1000
avg_time_per_doc = processing_time / len(documents)

print("📊 Batch Processing Summary:")
print(f"  📚 Documents processed: {len(documents)}")
print(f"  🎯 Total entities found: {total_entities}")
print(f"  ⏱️  Total processing time: {processing_time:.2f}ms")
print(f"  📈 Average per document: {avg_time_per_doc:.2f}ms")
print(f"  🏃 Throughput: {len(documents) / (processing_time/1000):.1f} docs/sec")

if entity_counts:
    print(f"\n🏷️  Entity breakdown:")
    for entity_type, count in entity_counts.items():
        print(f"    {entity_type}: {count}")

## 8. Performance Showcase - See the Speed

Let's demonstrate DataFog's performance advantage with a realistic document:

In [ ]:
# Realistic business document (similar to what you'd process in production)
large_document = """
CONFIDENTIAL EMPLOYEE REPORT - Q1 2024

=== EXECUTIVE SUMMARY ===
Report generated by: Sarah Johnson (sarah.johnson@company.com)
Date: March 15, 2024
Department: Human Resources
Contact: (555) 100-HR00 ext. 1234

=== EMPLOYEE RECORDS ===

1. John Smith (ID: EMP-001)
   Email: john.smith@company.com
   Phone: (555) 123-4567
   SSN: 123-45-6789
   Address: 123 Oak Street, San Francisco, CA 94102
   Manager: David Chen (david.chen@company.com)
   Salary: $85,000 annually
   Start Date: January 15, 2020

2. Maria Rodriguez (ID: EMP-002)
   Email: maria.rodriguez@company.com
   Phone: (555) 987-6543
   SSN: 987-65-4321
   Address: 456 Pine Ave, Los Angeles, CA 90210
   Manager: Lisa Wang (lisa.wang@company.com)
   Emergency Contact: Carlos Rodriguez (555) 111-2233

3. Michael Johnson (ID: EMP-003)
   Email: michael.j@company.com
   Personal Email: mike.personal@gmail.com
   Phone: (555) 456-7890
   SSN: 456-78-9012
   Credit Card on file: 4532-1234-5678-9012 (expires 12/26)
   
=== PAYROLL INFORMATION ===
Bank routing: 123456789
Direct deposit accounts verified on 2024-03-01
Tax ID: 12-3456789

=== CONTACT INFORMATION ===
HR Helpline: (555) 888-4HR7
Benefits questions: benefits@company.com
IT Support: support@company.com
Office address: 789 Corporate Blvd, Suite 100, Business City, NY 10001

This document contains sensitive employee information and should be handled according to 
company privacy policies and applicable laws including GDPR, CCPA, and HIPAA where applicable.

Report ID: RPT-2024-Q1-001
Classification: CONFIDENTIAL
Retention: 7 years from creation date
"""

print("🚀 Performance Benchmark\n")
print(f"📄 Document size: {len(large_document):,} characters")
print(f"📝 Lines of text: {len(large_document.splitlines())}")
print("=" * 60)

# Test with different engines
engines_to_test = [
    ("regex", "🚀 Regex Engine (Fastest)"),
    ("smart", "🧠 Smart Engine (Balanced)"),
]

results_comparison = {}

for engine_name, description in engines_to_test:
    try:
        print(f"\n{description}")
        print("-" * 30)
        
        service = TextService(engine=engine_name)
        
        # Run multiple times for accurate timing
        times = []
        for _ in range(3):
            start = time.time()
            result = service.annotate_text_sync(large_document)
            end = time.time()
            times.append((end - start) * 1000)
        
        avg_time = sum(times) / len(times)
        
        # Count entities found
        total_entities = sum(len(entities) for entities in result.values() if entities)
        entity_types = len([k for k, v in result.items() if v])
        
        results_comparison[engine_name] = {
            'time': avg_time,
            'entities': total_entities,
            'types': entity_types
        }
        
        print(f"⏱️  Average time: {avg_time:.2f}ms")
        print(f"🎯 Entities found: {total_entities}")
        print(f"🏷️  Entity types: {entity_types}")
        print(f"📊 Throughput: {len(large_document) / (avg_time/1000):,.0f} chars/sec")
        
    except Exception as e:
        print(f"❌ {engine_name} not available: {e}")

# Performance comparison
if len(results_comparison) > 1:
    print("\n🏆 Performance Comparison:")
    print("=" * 40)
    
    fastest_time = min(r['time'] for r in results_comparison.values())
    
    for engine, stats in results_comparison.items():
        speedup = fastest_time / stats['time'] if stats['time'] > 0 else 1
        if speedup >= 1:
            print(f"{engine}: {stats['time']:.2f}ms ({speedup:.1f}x faster) - {stats['entities']} entities")
        else:
            slowdown = stats['time'] / fastest_time
            print(f"{engine}: {stats['time']:.2f}ms ({slowdown:.1f}x slower) - {stats['entities']} entities")

print("\n✅ DataFog delivers production-ready performance for real-world documents!")

## 🎉 Congratulations!

You've completed the DataFog quick start guide! Here's what you've learned:

### ✅ Key Takeaways

1. **🚀 Speed**: DataFog is 190x faster than traditional NLP for structured PII
2. **🧠 Intelligence**: GLiNER and smart cascading provide excellent accuracy
3. **🔒 Flexibility**: Multiple anonymization options (redact, replace, hash)
4. **🎯 Precision**: Target specific PII types for selective processing
5. **📚 Scale**: Efficient batch processing for production workloads

### 🛠️ Engine Selection Guide

| Engine | Best For | Speed | Accuracy |
|--------|----------|-------|----------|
| `regex` | Structured PII (emails, phones, SSN) | 🚀🚀🚀 | ⭐⭐⭐ |
| `gliner` | Named entities (people, orgs, locations) | 🚀🚀 | ⭐⭐⭐⭐ |
| `smart` | **Production use (recommended)** | 🚀🚀 | ⭐⭐⭐⭐⭐ |

### 🚀 Next Steps

- **Production**: Use `engine="smart"` for best balance of speed and accuracy
- **High Volume**: Use `engine="regex"` for maximum speed on structured data
- **Custom Entities**: Explore GLiNER models for specialized use cases
- **Integration**: Check out our [documentation](https://docs.datafog.ai) for API details

### 💬 Get Help

- 📖 [Documentation](https://docs.datafog.ai)
- 💬 [Discord Community](https://discord.gg/bzDth394R4)
- 🐛 [GitHub Issues](https://github.com/datafog/datafog-python/issues)
- 📧 [Contact Us](mailto:hi@datafog.ai)

**Happy data processing with DataFog! 🌟**